# Bronze — Carga de JSON para Delta

Lê os arquivos JSON depositados pelo GitHub Actions no Volume,  
faz merge na tabela `judicial.bronze.datajud_raw` e move os arquivos para `processados/`.

**Pré-requisito:** `setup_environment` executado e arquivos presentes em `/Volumes/judicial/bronze/raw_files/{tribunal}/`

## 1. Imports e configuração

In [ ]:
import os
import shutil

from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit

VOLUME_BASE  = "/Volumes/judicial/bronze/raw_files"
BRONZE_TABLE = "judicial.bronze.datajud_raw"
TRIBUNAIS    = ["TJSC", "TJPR"]

## 2. Carga e merge por tribunal

In [ ]:
for tribunal in TRIBUNAIS:
    source_dir    = f"{VOLUME_BASE}/{tribunal}"
    processed_dir = f"{VOLUME_BASE}/processados/{tribunal}"

    # ── Verificar arquivos disponíveis ────────────────────────────────────
    try:
        arquivos = [f for f in os.listdir(source_dir) if f.endswith(".json")]
    except FileNotFoundError:
        print(f"{tribunal}: diretório não encontrado, pulando")
        continue

    if not arquivos:
        print(f"{tribunal}: nenhum arquivo para processar")
        continue

    print(f"\n▶ {tribunal} — {len(arquivos)} arquivo(s) encontrado(s)")

    # ── Ler JSONs como DataFrame ──────────────────────────────────────────
    df = (
        spark.read
        .option("multiLine", True)
        .json(source_dir)
        .withColumn("tribunal",      lit(tribunal))
        .withColumn("data_ingestao", current_timestamp())
    )

    # ── Criar tabela ou fazer merge ───────────────────────────────────────
    if not spark.catalog.tableExists(BRONZE_TABLE):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("tribunal")
            .saveAsTable(BRONZE_TABLE)
        )
        print(f"  Tabela criada: {df.count():,} registros")
    else:
        (
            DeltaTable.forName(spark, BRONZE_TABLE)
            .alias("t")
            .merge(df.alias("s"), "t._id = s._id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"  Merge concluído: {df.count():,} registros")

    # ── Mover arquivos para processados/ ─────────────────────────────────
    os.makedirs(processed_dir, exist_ok=True)
    for arquivo in arquivos:
        shutil.move(f"{source_dir}/{arquivo}", f"{processed_dir}/{arquivo}")
        print(f"  Movido: {arquivo} → processados/{tribunal}/")

## 3. Verificação

In [ ]:
%sql
SELECT tribunal, COUNT(*) AS total, MAX(data_ingestao) AS ultima_ingestao
FROM judicial.bronze.datajud_raw
GROUP BY tribunal
ORDER BY tribunal